# 01 — Oslo Børs data overview

Run `uv run python EDA/build_dataset.py` before this notebook to generate `Data/processed/` from `Data/raw/`.

This notebook introduces the repository's equity dataset prior to [02_sector_dependence.ipynb](02_sector_dependence.ipynb):

1. **Raw TITLON export (`Data/raw/`)**: dimensions, date span, all 49 columns, and missingness.
2. **Canonical processed contract (`Data/processed/`)**: files, shapes, date bounds, and ICB industry breakdown.
3. **Price vs. `AdjustedPrice`**: why adjusted prices are required (splits, ISIN changes, and dividends), plus indexed 10-year price trajectories across industries.
4. **Missingness, short-gap interpolation, and event corrections**: per-stock missingness, gap-length distribution, availability heatmaps, a zoomed interpolation example, and the two reviewed corporate-action corrections.
5. **Log-return sanity checks**: cross-sectional summary statistics, fat tails, annualized volatility, extreme days, and a quick template for slicing train/validation/test splits in downstream experiments.

In [ ]:
from pathlib import Path
import json

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Data/processed/manifest.json").is_file())
RAW_DIR = ROOT / "Data/raw"
PROCESSED_DIR = ROOT / "Data/processed"

RAW_PRICES_PATH = RAW_DIR / "BMG9156K1018_10.09.2026_14.59.34.csv"
SECTOR_MAPPING_PATH = RAW_DIR / "stock_sector_mapping.csv"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 20)
pd.set_option("display.precision", 4)

manifest = json.loads((PROCESSED_DIR / "manifest.json").read_text())
stocks = pd.read_csv(PROCESSED_DIR / "stocks.csv").set_index("stock_id", verify_integrity=True)
prices_obs = (
    pd.read_csv(PROCESSED_DIR / "prices_observed.csv", parse_dates=["Date"], float_precision="round_trip")
    .set_index("Date")
    .sort_index()
)
log_ret_obs = (
    pd.read_csv(PROCESSED_DIR / "log_returns_observed.csv", parse_dates=["Date"], float_precision="round_trip")
    .set_index("Date")
    .sort_index()
)
log_ret_interp = (
    pd.read_csv(PROCESSED_DIR / "log_returns_interpolated.csv", parse_dates=["Date"], float_precision="round_trip")
    .set_index("Date")
    .sort_index()
)

print("Loaded Data/processed/ successfully.")
print(f"Stocks: {len(stocks)} | Price sessions: {len(prices_obs):,} ({prices_obs.index[0].date()} .. {prices_obs.index[-1].date()})")
print(f"Return sessions: {len(log_ret_interp):,} ({log_ret_interp.index[0].date()} .. {log_ret_interp.index[-1].date()})")

## 1. Raw TITLON export (`Data/raw/`)

Inspect the raw equity export (`BMG9156K1018_10.09.2026_14.59.34.csv`) in chunks so memory stays low: total row count, distinct ISINs/symbols/companies, date coverage, and missingness across all 49 columns.

In [ ]:
chunk_iter = pd.read_csv(
    RAW_PRICES_PATH,
    sep=";",
    dtype="string",
    keep_default_na=True,
    na_values=["", "NA", "NaN"],
    chunksize=250_000,
)

total_rows = 0
non_null_counts = None
sample_values = None
min_date, max_date = None, None
unique_isins, unique_companies, unique_currencies, unique_exchanges = set(), set(), set(), set()

for chunk in chunk_iter:
    total_rows += len(chunk)
    counts = chunk.notna().sum()
    non_null_counts = counts if non_null_counts is None else non_null_counts.add(counts)
    if sample_values is None:
        sample_values = chunk.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
    c_min, c_max = chunk["Date"].min(), chunk["Date"].max()
    min_date = c_min if min_date is None or c_min < min_date else min_date
    max_date = c_max if max_date is None or c_max > max_date else max_date
    unique_isins.update(chunk["ISIN"].dropna().unique())
    unique_companies.update(chunk["CompanyId"].dropna().unique())
    unique_currencies.update(chunk["Currency"].dropna().unique())
    unique_exchanges.update(chunk["Exchange"].dropna().unique())

raw_summary = pd.Series({
    "File": RAW_PRICES_PATH.name,
    "Rows": f"{total_rows:,}",
    "Columns": len(non_null_counts),
    "Date span": f"{min_date} to {max_date}",
    "Distinct ISINs": f"{len(unique_isins):,}",
    "Distinct CompanyIds": f"{len(unique_companies):,}",
    "Currencies": ", ".join(sorted(unique_currencies)),
    "Exchanges": ", ".join(sorted(unique_exchanges)),
}, name="Raw TITLON equity export")
display(raw_summary.to_frame())

column_overview = pd.DataFrame({
    "column": non_null_counts.index,
    "non_null": non_null_counts.values.astype(int),
    "missing": (total_rows - non_null_counts.values).astype(int),
    "missing_pct": 100.0 * (total_rows - non_null_counts.values) / total_rows,
    "example_value": sample_values.values,
})
with pd.option_context("display.max_rows", 60):
    display(column_overview)

## 2. Canonical processed contract (`Data/processed/`)

`EDA/build_dataset.py` filters the raw export to the 10-year XOSL calendar, attaches Euronext ICB classifications from `Data/raw/stock_sector_mapping.csv`, links nonoverlapping ISIN transitions, selects one share class per issuer, applies the thesis eligibility screen (`last price valid`, `< 5%` missing, max gap `<= 5` sessions), and writes wide `Date × stock_id` CSVs in log-return form.

In [ ]:
contract_table = pd.DataFrame([
    {
        "file": "stocks.csv",
        "rows": len(stocks),
        "cols": stocks.shape[1] + 1,
        "date_span": "metadata (1 row / stock)",
        "missing_cells": int(stocks.isna().sum().sum()),
        "purpose": "stock_id -> name, issuer, ICB hierarchy, missingness & interpolation audit",
    },
    {
        "file": "prices_observed.csv",
        "rows": len(prices_obs),
        "cols": prices_obs.shape[1] + 1,
        "date_span": f"{prices_obs.index[0].date()} .. {prices_obs.index[-1].date()}",
        "missing_cells": int(prices_obs.isna().sum().sum()),
        "purpose": "Uninterpolated AdjustedPrice (NOK) on full XOSL session grid",
    },
    {
        "file": "log_returns_observed.csv",
        "rows": len(log_ret_obs),
        "cols": log_ret_obs.shape[1] + 1,
        "date_span": f"{log_ret_obs.index[0].date()} .. {log_ret_obs.index[-1].date()}",
        "missing_cells": int(log_ret_obs.isna().sum().sum()),
        "purpose": "Observed daily log returns + reviewed event corrections (for strict out-of-sample evaluation)",
    },
    {
        "file": "log_returns_interpolated.csv",
        "rows": len(log_ret_interp),
        "cols": log_ret_interp.shape[1] + 1,
        "date_span": f"{log_ret_interp.index[0].date()} .. {log_ret_interp.index[-1].date()}",
        "missing_cells": int(log_ret_interp.isna().sum().sum()),
        "purpose": "Short-gap spline-interpolated daily log returns + reviewed event corrections",
    },
])
display(contract_table)

print("Contract metadata from Data/processed/manifest.json:")
print(json.dumps(manifest["contract"], indent=2))
print("\nSample rows from stocks.csv:")
display(stocks[["name", "icb_industry_name", "icb_supersector_name", "missing_prices", "estimated_prices", "remaining_missing_returns"]].head(10))

In [ ]:
industry_counts = stocks["icb_industry_name"].value_counts().sort_values(ascending=True)
supersector_counts = stocks["icb_supersector_name"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

bars = axes[0].barh(industry_counts.index, industry_counts.values, color="#4477AA")
axes[0].bar_label(bars, padding=3, fontsize=8.5)
axes[0].set(
    title=f"Sample composition by ICB Industry (n = {len(stocks)})",
    xlabel="Number of stocks",
    xlim=(0, industry_counts.max() + 4),
)
axes[0].grid(axis="y", visible=False)

top_ss = supersector_counts.head(12).iloc[::-1]
bars2 = axes[1].barh(top_ss.index, top_ss.values, color="#66AA55")
axes[1].bar_label(bars2, padding=3, fontsize=8.5)
axes[1].set(
    title=f"Top 12 ICB Supersectors ({len(supersector_counts)} total)",
    xlabel="Number of stocks",
    xlim=(0, top_ss.max() + 4),
)
axes[1].grid(axis="y", visible=False)

plt.show()

## 3. Recorded `Price` vs. `AdjustedPrice` and 10-year trajectories

Unadjusted share prices jump mechanically on stock splits and drift downward relative to total shareholder wealth as cash dividends are paid out. Comparing TITLON's raw `Price` against `AdjustedPrice = Price × CorpAdj × DividendAdj` shows both effects clearly:

- **Splits + ISIN transitions:** Kongsberg Gruppen (5-for-1 split on `2025-06-03`) and Tomra Systems (2-for-1 split on `2022-05-27`).
- **Cumulative dividends:** Equinor and DNB Bank over the full 10-year window.

In [ ]:
# Load only the four showcase companies from the raw export to compare Price vs AdjustedPrice.
showcase = [
    ("Kongsberg Gruppen (split + ISIN change)", ["NO0003043309", "NO0013536151"], "2025-05-01", "2025-07-01", "2025-06-03"),
    ("Tomra Systems (split + ISIN change)", ["NO0005668905", "NO0012470089"], "2022-04-25", "2022-06-25", "2022-05-27"),
    ("Equinor (10y dividend adjustment)", ["NO0010096985"], "2016-09-08", "2026-09-08", None),
    ("DNB Bank (10y dividend adjustment)", ["NO0010161896"], "2016-09-08", "2026-09-08", None),
]
needed_isins = {isin for _, isins, _, _, _ in showcase for isin in isins}

parts = []
for chunk in pd.read_csv(
    RAW_PRICES_PATH,
    sep=";",
    decimal=",",
    usecols=["Date", "ISIN", "Price", "AdjustedPrice"],
    parse_dates=["Date"],
    chunksize=250_000,
):
    sub = chunk.loc[chunk["ISIN"].isin(needed_isins)]
    if not sub.empty:
        parts.append(sub)
raw_showcase = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["Date", "ISIN"])

fig, axes = plt.subplots(2, 2, figsize=(13, 7), constrained_layout=True)
for ax, (title, isins, start, end, event_date) in zip(axes.flat, showcase):
    sub = (
        raw_showcase.loc[
            raw_showcase["ISIN"].isin(isins)
            & raw_showcase["Date"].between(pd.Timestamp(start), pd.Timestamp(end))
        ]
        .dropna(subset=["Price", "AdjustedPrice"])
        .drop_duplicates(subset=["Date"])
        .sort_values("Date")
    )
    # Index both series to 100 at window start so relative paths are directly comparable.
    p_idx = 100.0 * sub["Price"] / sub["Price"].iloc[0]
    a_idx = 100.0 * sub["AdjustedPrice"] / sub["AdjustedPrice"].iloc[0]
    ax.plot(sub["Date"], p_idx, color="#5B6575", lw=1.4, label="Recorded Price (start = 100)")
    ax.plot(sub["Date"], a_idx, color="#0072B2", lw=1.6, label="AdjustedPrice (start = 100)")
    if event_date is not None:
        ax.axvline(pd.Timestamp(event_date), color="#D55E00", linestyle="--", lw=1.0, label=f"Split ({event_date})")
    ax.set(title=title, ylabel="Indexed level (start = 100)")
    ax.legend(fontsize=8, frameon=False, loc="best")
    ax.tick_params(axis="x", rotation=20)

plt.show()

In [ ]:
# Indexed 10-year AdjustedPrice trajectories for eight liquid Oslo Bors stocks across industries.
panel_examples = [
    ("NO0013536151", "Kongsberg Gruppen (Industrials)"),
    ("NO0010096985", "Equinor (Energy)"),
    ("NO0010161896", "DNB Bank (Financials)"),
    ("NO0005052605", "Norsk Hydro (Basic Materials)"),
    ("NO0012470089", "Tomra Systems (Industrials)"),
    ("CY0200352116", "Frontline (Industrials)"),
    ("NO0003054108", "Mowi (Consumer Staples)"),
    ("NO0010063308", "Telenor (Telecommunications)"),
]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), constrained_layout=True)
for sid, label in panel_examples:
    series = prices_obs[sid]
    indexed = 100.0 * series / series.iloc[0]
    axes[0].plot(prices_obs.index, indexed, lw=1.25, label=label)
    axes[1].plot(prices_obs.index, indexed, lw=1.25, label=label)

axes[0].axhline(100, color="0.5", ls=":", lw=0.9)
axes[0].set(title="10-year AdjustedPrice trajectories (linear scale)", ylabel="Indexed AdjustedPrice (Sep 2016 = 100)")
axes[1].axhline(100, color="0.5", ls=":", lw=0.9)
axes[1].set_yscale("log")
axes[1].set(title="Same trajectories (log scale)", ylabel="Indexed AdjustedPrice (log scale)")
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0].legend(fontsize=7.5, frameon=False, loc="upper left")
plt.show()

## 4. Missingness, gap structure, and short-gap interpolation

Even after the 111-stock eligibility screen, 48 of the 111 stocks have at least one missing trading-day price (1,225 missing price cells in total; 63 stocks have zero missing prices).

- Internal gaps of 1–5 trading sessions are filled with positive natural cubic splines (1,222 prices -> 2,259 daily log returns).
- **Remaining missing values after interpolation**:
  - 3 prices missing on the very first trading day (`2016-09-08`), leaving 3 missing returns on `2016-09-09`.
  - 1 return explicitly set to `NaN` via reviewed event corrections (`EAM Solar`, `2024-10-31`).

In [ ]:
# Gap-length distribution across all 111 observed price histories.
gap_rows = []
for sid in prices_obs.columns:
    vals = prices_obs[sid].to_numpy()
    changes = np.diff((~np.isfinite(vals)).astype(int), prepend=0, append=0)
    for first, stop in zip(np.flatnonzero(changes == 1), np.flatnonzero(changes == -1)):
        gap_rows.append({
            "stock_id": sid,
            "name": stocks.at[sid, "name"],
            "start": prices_obs.index[first],
            "end": prices_obs.index[stop - 1],
            "length": int(stop - first),
            "internal": bool(first > 0 and stop < len(vals)),
        })
gaps_df = pd.DataFrame(gap_rows)

missing_overview = pd.DataFrame([
    ("Stocks with 0 missing prices", int((stocks["missing_prices"] == 0).sum())),
    ("Stocks with >= 1 missing price", int((stocks["missing_prices"] > 0).sum())),
    ("Observed missing price cells", int(prices_obs.isna().sum().sum())),
    ("Interpolated price cells (internal gaps <= 5)", int(stocks["estimated_prices"].sum())),
    ("Remaining missing price cells (boundary: 2016-09-08)", int(stocks["remaining_missing_prices"].sum())),
    ("Observed missing return cells", int(log_ret_obs.isna().sum().sum())),
    ("Interpolated return cells", int((log_ret_obs.isna() & log_ret_interp.notna()).sum().sum())),
    ("Remaining missing return cells", int(log_ret_interp.isna().sum().sum())),
    ("Complete return dates (all 111 stocks): observed", int(log_ret_obs.notna().all(axis=1).sum())),
    ("Complete return dates (all 111 stocks): interpolated", int(log_ret_interp.notna().all(axis=1).sum())),
], columns=["Metric", "Value"])
display(missing_overview)

print("Top 10 stocks by observed missing price sessions:")
display(
    stocks.sort_values("missing_prices", ascending=False)[
        ["name", "icb_industry_name", "absent_records", "recorded_unavailable_prices",
         "missing_prices", "estimated_prices", "remaining_missing_prices", "remaining_missing_returns"]
    ].head(10)
)

print("Reviewed corporate-action return corrections (from manifest.json):")
display(pd.DataFrame(manifest["return_corrections"]))

print("Remaining missing cells in log_returns_interpolated.csv:")
rem = log_ret_interp.isna().stack()
rem_pairs = rem[rem].index.to_frame(index=False, name=["Date", "stock_id"])
rem_pairs["name"] = rem_pairs["stock_id"].map(stocks["name"])
display(rem_pairs)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0), constrained_layout=True)

# Panel A: Gap-length counts
gap_counts = gaps_df.groupby(["length", "internal"]).size().unstack(fill_value=0)
gap_counts.plot(kind="bar", stacked=True, ax=axes[0], color=["#CC6677", "#4477AA"])
axes[0].set(title="Price gap lengths (all 111 stocks)", xlabel="Consecutive missing sessions", ylabel="Number of gaps")
axes[0].legend(["Boundary (unfilled)", "Internal (spline-filled)"], frameon=False, fontsize=8)
axes[0].tick_params(axis="x", rotation=0)

# Panel B: Missing prices per stock distribution
miss_per_stock = stocks["missing_prices"]
axes[1].hist(miss_per_stock[miss_per_stock > 0], bins=18, color="#4477AA", edgecolor="white")
axes[1].set(
    title=f"Missing prices among the {(miss_per_stock > 0).sum()} stocks with gaps",
    xlabel="Missing trading sessions (out of 2,510)",
    ylabel="Number of stocks",
)

# Panel C: Zoom into one multi-day internal gap to visualize spline interpolation
example_gap = gaps_df.loc[gaps_df["internal"] & gaps_df["length"].ge(4)].iloc[0]
sid = example_gap["stock_id"]
pos = prices_obs.index.get_loc(example_gap["start"])
window_slice = slice(max(0, pos - 6), min(len(prices_obs), pos + example_gap["length"] + 7))
sub_obs = prices_obs[sid].iloc[window_slice]
# Reconstruct implied interpolated prices over the local window from log_returns_interpolated
anchor_idx = np.flatnonzero(sub_obs.notna())[0]
sub_rets = log_ret_interp[sid].reindex(sub_obs.index).iloc[anchor_idx + 1:]
reconstructed = pd.Series(index=sub_obs.index, dtype=float)
reconstructed.iloc[anchor_idx] = sub_obs.iloc[anchor_idx]
reconstructed.iloc[anchor_idx + 1:] = sub_obs.iloc[anchor_idx] * np.exp(sub_rets.cumsum())
imputed_pts = reconstructed.loc[sub_obs.isna()]

axes[2].plot(reconstructed.index, reconstructed.values, color="#4477AA", lw=1.3, alpha=0.8, label="Interpolated path")
axes[2].scatter(sub_obs.dropna().index, sub_obs.dropna().values, color="black", s=24, zorder=4, label="Observed price")
axes[2].scatter(imputed_pts.index, imputed_pts.values, color="#D55E00", s=34, marker="x", lw=1.8, zorder=5, label="Spline-filled")
axes[2].set(
    title=f"Example {example_gap['length']}-day gap: {example_gap['name']}",
    ylabel="AdjustedPrice (NOK)",
)
axes[2].tick_params(axis="x", rotation=25)
axes[2].legend(fontsize=8, frameon=False)

plt.show()

In [ ]:
# Availability heatmaps across the full 10-year sample (black = available, white = missing).
order = stocks.sort_values(["missing_prices", "name"], ascending=[False, True]).index
fig, axes = plt.subplots(2, 1, figsize=(13.5, 6.2), sharex=True, constrained_layout=True)

for ax, frame, title in [
    (axes[0], prices_obs[order], "Observed AdjustedPrice availability (sorted by missingness; white = missing)"),
    (axes[1], log_ret_interp[order], "Interpolated daily log-return availability (white = 4 remaining NaNs)"),
]:
    ax.imshow(frame.notna().T, aspect="auto", interpolation="nearest", cmap="Greys", vmin=0, vmax=1)
    ax.set(title=title, ylabel="Stocks (111)")
    ax.grid(False)

ticks = np.linspace(0, len(log_ret_interp) - 1, 9).astype(int)
axes[1].set_xticks(ticks, log_ret_interp.index[ticks].strftime("%Y-%m"))
axes[1].set_xlabel("Exchange session")
plt.show()

## 5. Daily log-return distribution sanity checks

Check cross-sectional summary statistics across all 111 stocks (annualized return/volatility, skewness, excess kurtosis, zero-return share, and extreme daily moves) and confirm that short-gap interpolation barely shifts marginal return statistics.

In [ ]:
def stock_stats(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "name": stocks.loc[frame.columns, "name"],
        "industry": stocks.loc[frame.columns, "icb_industry_name"],
        "n_returns": frame.count(),
        "ann_mean_log": frame.mean() * 252,
        "ann_vol": frame.std(ddof=1) * np.sqrt(252),
        "skewness": frame.skew(),
        "excess_kurtosis": frame.kurt(),
        "zero_frac": frame.eq(0).sum() / frame.count(),
        "min_day": frame.min(),
        "max_day": frame.max(),
    })

stats_obs = stock_stats(log_ret_obs)
stats_interp = stock_stats(log_ret_interp)

cross_section = pd.concat({
    "Observed (median)": stats_obs[["ann_mean_log", "ann_vol", "skewness", "excess_kurtosis", "zero_frac"]].median(),
    "Interpolated (median)": stats_interp[["ann_mean_log", "ann_vol", "skewness", "excess_kurtosis", "zero_frac"]].median(),
    "Interpolated (IQR low)": stats_interp[["ann_mean_log", "ann_vol", "skewness", "excess_kurtosis", "zero_frac"]].quantile(0.25),
    "Interpolated (IQR high)": stats_interp[["ann_mean_log", "ann_vol", "skewness", "excess_kurtosis", "zero_frac"]].quantile(0.75),
}, axis=1)
display(cross_section)

print("10 largest absolute daily log returns in log_returns_interpolated.csv:")
stacked = log_ret_interp.rename_axis(columns="stock_id").stack().rename("log_return").reset_index()
stacked["name"] = stacked["stock_id"].map(stocks["name"])
stacked["industry"] = stacked["stock_id"].map(stocks["icb_industry_name"])
stacked["simple_return_pct"] = 100.0 * np.expm1(stacked["log_return"])
extremes = stacked.loc[stacked["log_return"].abs().nlargest(10).index, ["Date", "stock_id", "name", "industry", "log_return", "simple_return_pct"]]
display(extremes.reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0), constrained_layout=True)

# Panel A: Pooled standardized daily log returns vs Normal(0, 1)
z = (log_ret_interp - log_ret_interp.mean()) / log_ret_interp.std(ddof=1)
z_flat = z.to_numpy().ravel()
z_flat = z_flat[np.isfinite(z_flat)]
grid = np.linspace(-6, 6, 400)
axes[0].hist(np.clip(z_flat, -6, 6), bins=80, density=True, color="#4477AA", alpha=0.65, label="Empirical (clipped ±6σ)")
axes[0].plot(grid, stats.norm.pdf(grid), color="#D55E00", lw=1.6, ls="--", label="Standard normal")
axes[0].set_yscale("log")
axes[0].set(title="Standardized daily log returns (log density)", xlabel="z-score", ylabel="Density", ylim=(1e-4, 1.0))
axes[0].legend(fontsize=8, frameon=False)

# Panel B: Cross-sectional annualized volatility
axes[1].hist(100.0 * stats_interp["ann_vol"], bins=22, color="#4477AA", edgecolor="white")
axes[1].axvline(100.0 * stats_interp["ann_vol"].median(), color="#D55E00", ls="--", lw=1.3,
                label=f"Median: {100.0 * stats_interp['ann_vol'].median():.1f}%")
axes[1].set(title="Annualized volatility across 111 stocks", xlabel="Annualized volatility (%)", ylabel="Number of stocks")
axes[1].legend(fontsize=8, frameon=False)

# Panel C: Cross-sectional zero-return fraction (stale / thinly traded check)
axes[2].hist(100.0 * stats_obs["zero_frac"], bins=22, color="#5B6575", edgecolor="white")
axes[2].axvline(100.0 * stats_obs["zero_frac"].median(), color="#D55E00", ls="--", lw=1.3,
                label=f"Median: {100.0 * stats_obs['zero_frac'].median():.1f}%")
axes[2].set(title="Zero-return session share (observed)", xlabel="Zero-return sessions (%)", ylabel="Number of stocks")
axes[2].legend(fontsize=8, frameon=False)

plt.show()

## 6. Quick usage pattern for downstream experiments

Every experiment can slice directly from `Data/processed/` without repeating data cleaning. For strict out-of-sample risk evaluation, evaluate on `log_returns_observed.csv` (realized market returns only) and fit any standardization on the training slice alone:

In [ ]:
# Example: select Energy stocks with zero missing prices and slice chronological 70/15/15 splits.
energy_ids = stocks.loc[
    stocks["icb_industry_name"].eq("Energy") & stocks["missing_prices"].eq(0)
].index.tolist()

panel = log_ret_obs[energy_ids]
n = len(panel)
n_train, n_val = round(0.70 * n), round(0.15 * n)
train_log = panel.iloc[:n_train]
val_log = panel.iloc[n_train:n_train + n_val]
test_log = panel.iloc[n_train + n_val:]

print(f"Selected {len(energy_ids)} complete Energy stocks:")
print(f"  train     : {train_log.index[0].date()} .. {train_log.index[-1].date()} ({len(train_log)} sessions)")
print(f"  validation: {val_log.index[0].date()} .. {val_log.index[-1].date()} ({len(val_log)} sessions)")
print(f"  test      : {test_log.index[0].date()} .. {test_log.index[-1].date()} ({len(test_log)} sessions)")